# 01 · XDF → CSV

Convert the raw LSL recordings into one wide CSV per session.

| | |
|---|---|
| **input**  | `data/raw/<subject>/*.xdf` |
| **output** | `data/csv/<subject>/<session>.csv` |

Each `.xdf` bundles three streams captured together — EEG (`EmotivDataStream-EEG`), surface EMG (`EMG_Stream`) and motion capture (`OptiTrack_BiomechIDs`). They are merged on the shared LSL timestamp; columns are prefixed by stream type (`EEG_`, `EMG_`, `Markers_`).

> The experiment data is **not distributed** (participant privacy). Place your own recordings under `data/raw/` to run this notebook.

In [ ]:
import sys
from pathlib import Path

# make the `motion_intent` package importable when running from notebooks/
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from motion_intent import config

In [ ]:
from motion_intent.io_xdf import summarize_xdf, xdf_to_dataframe, export_session_csv

## Inspect one recording

In [ ]:
xdf_files = sorted(config.RAW_DIR.rglob('*.xdf'))
print(f'{len(xdf_files)} xdf files under {config.RAW_DIR}')

if xdf_files:
    summarize_xdf(xdf_files[0])

## Convert every session

`export_session_csv` writes `<out_dir>/<same-stem>.csv`. Subject folder names are taken from the parent directory of each `.xdf`.

In [ ]:
for xdf_path in xdf_files:
    subject = xdf_path.parent.name
    out_dir = config.CSV_DIR / subject
    csv_path = export_session_csv(xdf_path, out_dir)
    print(csv_path.relative_to(config.DATA_DIR))

## Quick look at the merged frame

In [ ]:
if xdf_files:
    df = xdf_to_dataframe(xdf_files[0])
    print(df.shape)
    display(df.filter(regex='^(t_sec|EEG_Cz|EMG_|Markers_).*').head())